# Hippo Encoder Improvement Run

Focused Colab workflow for improving the tiny encoder and testing the two-rope variable-budget representation.

Use a GPU runtime. H100 is a good choice for larger batch sizes and faster teacher embedding, but the default values below are still conservative enough to debug safely.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os

if not os.path.exists('/content/Hippo-encoder/.git'):
    !git clone https://github.com/CameronBadman/Hippo-encoder.git

%cd /content/Hippo-encoder
!git pull --ff-only
!pip install -e .

In [ ]:
import json
import os
from pathlib import Path
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

PROJECT = Path('/content/Hippo-encoder')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_DIR = DRIVE_ROOT / 'hippo_encoder_data'
RUN_DIR = DRIVE_ROOT / 'hippo_encoder_runs'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

PAIR_DATA = DATA_DIR / 'train_pairs_all_nli_pair_100k.jsonl'
PAIR_RUN = RUN_DIR / 'distill-bge-small-all-nli-pair-100k'
PAIR_CONFIG = PROJECT / 'configs' / 'colab_all_nli_pair_h100.json'
REGION_CASES = DATA_DIR / 'region_cases_all_nli_pair_3000.json'
BENCH_DIR = RUN_DIR / 'rope_benchmarks'
BENCH_DIR.mkdir(parents=True, exist_ok=True)

print('pair data:', PAIR_DATA)
print('pair run:', PAIR_RUN)
print('region cases:', REGION_CASES)

## Build Pair Data

`all_nli_pair` was the best signal in the earlier logs. Start with 100k rows. On H100, raise `--limit` later if this works cleanly.

In [ ]:
%cd /content/Hippo-encoder
if not PAIR_DATA.exists():
    !python scripts/prepare_pair_dataset.py \
      --source all_nli_pair \
      --limit 100000 \
      --output {PAIR_DATA}
else:
    print('exists:', PAIR_DATA)

## Train Stronger Encoder

This writes checkpoints directly to Drive so the pair-trained model is not lost between Colab sessions. For H100, `batch_size=64` should usually be fine; if memory is tight, change it to `32`.

In [ ]:
config = {
    'teacher_model_name': 'intfloat/e5-base-v2',
    'student_model_name': 'BAAI/bge-small-en-v1.5',
    'dataset_jsonl': str(PAIR_DATA),
    'output_dir': str(PAIR_RUN),
    'max_text_length': 64,
    'batch_size': 64,
    'num_epochs': 3,
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'log_every': 20,
    'save_every': 100000,
    'num_workers': 2,
    'teacher_text_weight': 1.0,
    'hidden_state_weight': 0.2,
    'contrastive_weight': 0.4,
    'contrastive_temperature': 0.07,
    'normalize_targets': True,
    'gradient_clip_norm': 1.0,
    'warmup_steps': 100,
    'seed': 42,
}
PAIR_CONFIG.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(PAIR_CONFIG.read_text())

In [ ]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config {PAIR_CONFIG}

## Pick And Evaluate Checkpoint

Use the latest epoch with a `heads.pt`. This avoids the path problems from the previous runs.

In [ ]:
heads = sorted(PAIR_RUN.glob('epoch-*/heads.pt'))
print('\n'.join(str(path) for path in heads))
assert heads, f'No checkpoints found under {PAIR_RUN}'
STUDENT_CKPT = heads[-1].parent
print('selected:', STUDENT_CKPT)

In [ ]:
%cd /content/Hippo-encoder
!python scripts/eval_student_encoder.py \
  --student-checkpoint {STUDENT_CKPT}

## Build Region Cases

These cases are built from the same pair data. They are for rope/formula evaluation, not for dense prompt-delta training.

In [ ]:
%cd /content/Hippo-encoder
if not REGION_CASES.exists():
    !python scripts/prepare_region_cases.py \
      --input-jsonl {PAIR_DATA} \
      --output {REGION_CASES} \
      --num-cases 3000 \
      --positives-per-case 2 \
      --negatives-per-case 3 \
      --batch-size 128
else:
    print('exists:', REGION_CASES)

## Rope Budget Benchmarks

Run both point and formula programs. The main signal is whether teacher-side fidelity improves as budget increases, then whether the student anchor tracks that improvement.

In [ ]:
%cd /content/Hippo-encoder
POINT_OUT = BENCH_DIR / 'rope_point_pair_checkpoint.json'
!python scripts/benchmark_rope_region.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint {STUDENT_CKPT} \
  --program-type point \
  --budgets 16 32 64 128 | tee {POINT_OUT}

In [ ]:
%cd /content/Hippo-encoder
FORMULA_OUT = BENCH_DIR / 'rope_formula_pair_checkpoint.json'
!python scripts/benchmark_rope_region.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint {STUDENT_CKPT} \
  --program-type formula \
  --budgets 16 32 64 128 | tee {FORMULA_OUT}

## Larger Region Case Benchmark

Once the sample benchmark looks sane, run the same test on generated cases. This is slower but more meaningful.

In [ ]:
%cd /content/Hippo-encoder
FORMULA_REGION_OUT = BENCH_DIR / 'rope_formula_region_cases_pair_checkpoint.json'
!python scripts/benchmark_rope_region.py \
  --cases {REGION_CASES} \
  --student-checkpoint {STUDENT_CKPT} \
  --program-type formula \
  --budgets 16 32 64 128 | tee {FORMULA_REGION_OUT}

## Optional H100 Scale-Up

After the 100k row run works, rerun with a larger pair dataset and a new output directory. Keep the old checkpoint instead of overwriting it.

In [ ]:
# Example scale-up values. Run manually after the baseline cells are working.
# PAIR_DATA_LARGE = DATA_DIR / 'train_pairs_all_nli_pair_300k.jsonl'
# PAIR_RUN_LARGE = RUN_DIR / 'distill-bge-small-all-nli-pair-300k'
# Recommended first scale-up change:
#   --limit 300000
#   batch_size 64 or 96 on H100
#   keep num_epochs at 3 until eval says otherwise